# 04 — Datos originales: baseline, accuracy engañosa y fuga

## Pregunta que guía el notebook

Qué ocurre si el mismo algoritmo recibe agregados de datos originales sin el trabajo semántico del pipeline. Este baseline es necesario para atribuir cualquier cambio posterior a los datos, no a un modelo diferente.

In [ ]:
from pathlib import Path
import json, sys
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import seaborn as sns
sns.set_theme(style='whitegrid', palette='deep')
def find_root():
    for candidate in [Path.cwd(), Path.cwd().parent, Path('/content/retail_customer_intelligence')]:
        if (candidate / 'requirements.txt').exists():
            return candidate.resolve()
    raise FileNotFoundError('No se encontró la raíz del proyecto. Abre la carpeta retail_customer_intelligence, no sólo el archivo .ipynb.')
ROOT = find_root()
sys.path.insert(0, str(ROOT / 'src'))
from retail_pipeline import *
ensure_folders(ROOT)
print('Proyecto:', ROOT)

## Un baseline válido, pero deliberadamente limitado

El baseline conoce conteos, importes y recencia calculados sobre registros crudos. Mantendremos algoritmo, semilla, objetivo y partición temporal. La comparación final será justa, aunque este conjunto sea menos expresivo.

In [ ]:
table = pd.read_parquet(ROOT / "data/processed/modelo_original_minimo.parquet")
predictions, metrics, classifier, _ = train_models(table, ROOT, "original")
dummy = no_purchase_baseline(predictions.compra_futura)

scorecard = pd.DataFrame({
    "regla nula: nadie compra": dummy,
    "baseline con datos originales": metrics["clasificacion"],
}).T.drop(columns="matriz_confusion")
display(scorecard.round(3))
print("El umbral se eligió exclusivamente con agosto:", round(metrics["umbral_seleccionado_validacion"], 2))

## Por qué accuracy no es la métrica de la caso

Si casi todos los clientes no compran, la regla nadie compra tiene una accuracy alta pero recall y F1 de compradores iguales a cero. Una campaña necesita una lista que concentre compradores, no una métrica que premie ignorarlos.

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(12, 4))
for ax, matrix, title in zip(
    axes,
    [np.array(dummy["matriz_confusion"]), np.array(metrics["clasificacion"]["matriz_confusion"])],
    ["Regla nula: nadie compra", "Baseline: datos originales"],
):
    sns.heatmap(matrix, annot=True, fmt="d", cmap="Blues", ax=ax)
    ax.set(title=title, xlabel="predicción", ylabel="real")
plt.tight_layout()

display(pd.DataFrame([metrics["ranking_por_probabilidad"]]).drop(columns="criterio_ordenamiento").round(3))

## La tentación: fuga de información

Entrenaremos el mismo modelo con la variable que cuenta actividad posterior al corte. El resultado puede ser espectacular, pero no representa una decisión que pudiera haberse tomado a tiempo. Se etiqueta como inválido y jamás se usa para generar la campaña.

In [ ]:
leaky_table = pd.read_parquet(ROOT / "data/processed/modelo_con_fuga_didactica.parquet")
leaky_predictions, leaky_metrics, _, _ = train_models(leaky_table, ROOT, "fuga_didactica")

leak_scorecard = pd.DataFrame({
    "baseline válido": metrics["clasificacion"],
    "con fuga — INVÁLIDO": leaky_metrics["clasificacion"],
}).T.drop(columns="matriz_confusion")
display(leak_scorecard.round(3))
print("Columna prohibida:", "lineas_posteriores_al_corte")
print("Veredicto: una métrica alta no compensa información que no existía al decidir.")

## Inspeccionar errores antes de cambiar el modelo

Las métricas agregadas no muestran a quién se prioriza mal. Falsos positivos consumen presupuesto; falsos negativos dejan oportunidades fuera de la lista. Los revisamos sólo con variables previas al corte.

In [ ]:
review = predictions.merge(
    table[["CustomerID", "snapshot_date", "lineas_raw", "importe_neto_raw", "dias_desde_ultima_raw"]],
    on=["CustomerID", "snapshot_date"],
)
threshold = metrics["umbral_seleccionado_validacion"]
review["tipo_error"] = np.select(
    [
        review.compra_futura.eq(1) & review.probabilidad_compra.lt(threshold),
        review.compra_futura.eq(0) & review.probabilidad_compra.ge(threshold),
    ],
    ["falso negativo", "falso positivo"], default="acierto",
)
display(review.tipo_error.value_counts().rename_axis("resultado").to_frame("clientes"))
display(review.query("tipo_error != 'acierto'").sort_values("probabilidad_compra", ascending=False).head(10))

## Mensaje para clase

Una caso guiado honesta distingue un baseline válido, un clasificador nulo que hace ver alta la accuracy y una fuga que debe descartarse aunque luzca excelente.

Artefactos: predicciones y métricas del baseline, más un experimento de fuga claramente marcado como inválido.